# 00 — Coleta: Atlas Digital de Desastres (Ampliação 1)

**Objetivo:** marcar quais municípios do AM e do RS tiveram **seca/estiagem** ou **enchente** reconhecida pela Defesa Civil, e quando. Sem isso a base não separa mês de crise de mês normal.

**Fonte:** Atlas Digital de Desastres no Brasil (MIDR/Sedec, dados do S2iD), 1991–2025. Planilha baixada de https://atlasdigital.mdr.gov.br e salva **sem edição** em `data/raw/`.

**Saída:** `data/processed/atlas_eventos_am_rs_2022_2024.csv` (e `.parquet`), uma linha por município e registro de desastre.

**Responsável:** Ampliação 1 (Atlas / S2iD). Rodar com a **raiz do repositório** como diretório de trabalho (ou direto de `notebooks/`, que o notebook sobe um nível).

In [1]:
import json, datetime, pathlib
import pandas as pd

print(f"pandas {pd.__version__}")

RAIZ = pathlib.Path.cwd()
if RAIZ.name == "notebooks":          # permite rodar de dentro de notebooks/
    RAIZ = RAIZ.parent
BRUTOS   = RAIZ / "data" / "raw"
TRATADOS = RAIZ / "data" / "processed"
TRATADOS.mkdir(parents=True, exist_ok=True)
ARQ_PROV = RAIZ / "docs" / "proveniencia.csv"
print("Trabalhando em:", RAIZ)

def registrar(fonte, url, metodo, parametros, n_linhas, arquivo_bruto):
    """Mesma função do 00_coleta_ibge, com uma trava: não duplica a linha se o notebook for rodado de novo."""
    rel = str(pathlib.Path(arquivo_bruto).relative_to(RAIZ))
    if ARQ_PROV.exists() and ARQ_PROV.stat().st_size > 0:
        ja = pd.read_csv(ARQ_PROV)
        if ((ja["fonte"] == fonte) & (ja["arquivo_bruto"] == rel)).any():
            print(f"  (proveniência já tinha: {fonte})")
            return
    linha = pd.DataFrame([{
        "fonte": fonte, "url": url, "metodo": metodo,
        "parametros": json.dumps(parametros, ensure_ascii=False),
        "n_linhas": n_linhas, "arquivo_bruto": rel,
        "coletado_em": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
    }])
    cabecalho = not ARQ_PROV.exists() or ARQ_PROV.stat().st_size == 0
    linha.to_csv(ARQ_PROV, mode="a", header=cabecalho, index=False)
    print(f"  ✓ proveniência: {fonte} ({n_linhas} linhas)")

pandas 3.1.1
Trabalhando em: /home/leonardo_b/Coding/CD/eda/crise-saude-ufam


## 1. Configuração

O enunciado lista os tipos COBRADE com pontos (`1.4.1.1.0`). **No Atlas o código vem sem pontos, como inteiro** (`14110`), e é assim que o filtro abaixo usa.

**Decisão: incluir o 13214 (Chuvas Intensas) como enchente.** A enchente do RS de maio de 2024 foi registrada quase toda sob esse código, que não está na lista do enunciado. O Atlas o classifica no grupo de desastres hidrológicos, o mesmo das inundações, enxurradas e alagamentos. Só com os 5 códigos da lista, o RS 2024 ficaria com 56 municípios atingidos em vez de 469 (de 497), e a variável principal do modelo (quem estava em enchente) ficaria muito subestimada. Por isso o 13214 entra como `enchente`. Ele fica marcado com `cobrade_no_enunciado = False`, o que permite refazer a análise só com os 5 códigos originais, como teste de sensibilidade.

In [2]:
ARQ_ATLAS = BRUTOS / "BD_Atlas_1991_2025_v1.1_2026.08.06_Consolidado.xlsx"
ABA       = "Atlas Valores Originais"      # danos humanos não mudam na aba "Corrigidos" (só valores em R$)
UFS       = ["AM", "RS"]
ANOS      = (2022, 2024)

# código COBRADE do Atlas (sem pontos) -> (grupo no projeto, está na lista do enunciado?)
COBRADE = {
    14110: ("seca",     True),    # Estiagem       (1.4.1.1.0)
    14120: ("seca",     True),    # Seca           (1.4.1.2.0)
    12100: ("enchente", True),    # Inundações     (1.2.1.0.0)
    12200: ("enchente", True),    # Enxurradas     (1.2.2.0.0)
    12300: ("enchente", True),    # Alagamentos    (1.2.3.0.0)
    13214: ("enchente", False),  # Chuvas Intensas — incluído por decisão
}
assert ARQ_ATLAS.exists(), f"Planilha não encontrada: {ARQ_ATLAS}. Confira o nome e se o Git LFS baixou o arquivo."

## 2. Leitura (a planilha tem ~76 mil linhas; leva cerca de 40 s)

In [3]:
atlas = pd.read_excel(ARQ_ATLAS, sheet_name=ABA, engine="openpyxl")
print(atlas.shape)
atlas[["Protocolo_S2iD","Nome_Municipio","Sigla_UF","Data_Registro","Data_Evento","Cod_Cobrade","Status","Cod_IBGE_Mun"]].head()

(76190, 70)


,Protocolo_S2iD,Nome_Municipio,Sigla_UF,Data_Registro,Data_Evento,Cod_Cobrade,Status,Cod_IBGE_Mun
0,SC-D-4215406-14110-19910107,Salto Veloso,SC,1991-01-07,1991-01-07,14110,Registro,4215406
1,RS-D-4313102-14110-19910112,Nova Palma,RS,1991-01-12,1991-01-12,14110,Registro,4313102
2,RS-D-4304952-14110-19910115,Caseiros,RS,1991-01-15,1991-01-15,14110,Registro,4304952
3,SC-D-4208609-14110-19910115,Jaborá,SC,1991-01-15,1991-01-15,14110,Registro,4208609
4,SC-D-4207650-14110-19910118,Iporã do Oeste,SC,1991-01-18,1991-01-18,14110,Registro,4207650


## 3. Filtro e tratamento

* UF = AM e RS; tipos da tabela `COBRADE`.
* Ano de 2022 a 2024 na **data do evento ou na data do registro** (um evento de dezembro/2021 registrado em janeiro/2022 ainda afeta os meses de referência de 2022).
* Mantemos `Status` (`Reconhecido` ou `Registro`) para a EDA decidir se usa só os reconhecidos.
* `codigo_ibge_6` = código de 7 dígitos sem o verificador, para juntar com o DATASUS.

In [4]:
d = atlas[atlas["Sigla_UF"].isin(UFS) & atlas["Cod_Cobrade"].isin(COBRADE)].copy()

a0, a1 = ANOS
na_janela = d["Data_Evento"].dt.year.between(a0, a1) | d["Data_Registro"].dt.year.between(a0, a1)
d = d[na_janela]

tab = pd.DataFrame({
    "codigo_ibge":   d["Cod_IBGE_Mun"].astype("int64"),
    "codigo_ibge_6": (d["Cod_IBGE_Mun"] // 10).astype("int64"),
    "nome_municipio": d["Nome_Municipio"],
    "uf":            d["Sigla_UF"],
    "protocolo_s2id": d["Protocolo_S2iD"],
    "status":        d["Status"],
    "data_registro": d["Data_Registro"].dt.date,
    "data_evento":   d["Data_Evento"].dt.date,
    "ano_mes_evento":   d["Data_Evento"].dt.strftime("%Y-%m"),
    "ano_mes_registro": d["Data_Registro"].dt.strftime("%Y-%m"),
    "cobrade":       d["Cod_Cobrade"],
    "tipologia":     d["descricao_tipologia"],
    "grupo":         d["Cod_Cobrade"].map(lambda c: COBRADE[c][0]),
    "cobrade_no_enunciado": d["Cod_Cobrade"].map(lambda c: COBRADE[c][1]),
    "afetados_seca_estiagem": d["DH_AFETADOS_SECA_ESTIAGEM"],
    "desabrigados":  d["DH_DESABRIGADOS"],
    "desalojados":   d["DH_DESALOJADOS"],
    "mortos":        d["DH_MORTOS"],
    "enfermos":      d["DH_ENFERMOS"],
    "danos_humanos_diretos": d["DH_total_danos_humanos_diretos"],
    "outros_afetados": d["DH_OUTROS AFETADOS"],
}).sort_values(["uf", "codigo_ibge", "data_evento"]).reset_index(drop=True)

print(tab.shape)
tab.head()

(2117, 21)


,codigo_ibge,codigo_ibge_6,nome_municipio,uf,protocolo_s2id,status,data_registro,data_evento,ano_mes_evento,ano_mes_registro,...,tipologia,grupo,cobrade_no_enunciado,afetados_seca_estiagem,desabrigados,desalojados,mortos,enfermos,danos_humanos_diretos,outros_afetados
0,1300029,130002,Alvarães,AM,AM-F-1300029-12100-20220606,Reconhecido,2022-06-09,2022-06-06,2022-06,2022-06,...,Inundações,enchente,True,0,88,40,0,1388,1639,4604
1,1300029,130002,Alvarães,AM,AM-F-1300029-14110-20221014,Reconhecido,2022-10-17,2022-10-14,2022-10,2022-10,...,Estiagem e Seca,seca,True,8868,0,0,0,1988,10856,8868
2,1300029,130002,Alvarães,AM,AM-F-1300029-14110-20230929,Reconhecido,2023-10-01,2023-09-29,2023-09,2023-10,...,Estiagem e Seca,seca,True,8032,0,0,0,4764,12928,8032
3,1300029,130002,Alvarães,AM,AM-F-1300029-14110-20240826,Reconhecido,2024-08-30,2024-08-26,2024-08,2024-08,...,Estiagem e Seca,seca,True,8288,0,0,0,2996,11418,8288
4,1300060,130006,Amaturá,AM,AM-F-1300060-12100-20220523,Reconhecido,2022-05-24,2022-05-23,2022-05,2022-05,...,Inundações,enchente,True,0,0,0,0,83,99,9348


## 4. Conferências

In [5]:
# 1) código de 7 dígitos e junção com as tabelas de municípios do projeto
assert tab["codigo_ibge"].astype(str).str.len().eq(7).all()
mun = pd.concat([pd.read_csv(TRATADOS / "municipios_am.csv"), pd.read_csv(TRATADOS / "municipios_rs.csv")])
fora = set(tab["codigo_ibge"]) - set(mun["municipio_id"])
print("Códigos do Atlas que não existem em municipios_am/rs.csv:", sorted(fora))   # esperado: []

# 2) registros por UF, grupo e ano do evento
print()
print(tab.groupby(["uf", "grupo", tab["ano_mes_evento"].str[:4]]).size().unstack(fill_value=0))

# 3) municípios distintos atingidos
print()
print(tab.groupby(["uf", "grupo"])["codigo_ibge"].nunique().rename("municipios_distintos"))

Códigos do Atlas que não existem em municipios_am/rs.csv: []

ano_mes_evento  2021  2022  2023  2024
uf grupo                              
AM enchente        0    64    11     5
   seca            0    22    62    61
RS enchente        0    40   552   587
   seca           22   322   364     5

uf  grupo   
AM  enchente     53
    seca         62
RS  enchente    478
    seca        447
Name: municipios_distintos, dtype: int64


In [6]:
# 4) o aviso do RS 2024: com e sem o código 13214
rs24 = tab[(tab["uf"] == "RS") & (tab["ano_mes_evento"].str[:4] == "2024") & (tab["grupo"] == "enchente")]
print("RS 2024, enchente — municípios só com os 5 códigos do enunciado:",
      rs24[rs24["cobrade_no_enunciado"]]["codigo_ibge"].nunique())
print("RS 2024, enchente — municípios incluindo 13214 (Chuvas Intensas):",
      rs24["codigo_ibge"].nunique(), "de 497")

# 5) distribuição por mês do evento (para ver o pico de maio/2024 no RS e set–out/2023 no AM)
print()
print(tab.groupby(["uf", "grupo", "ano_mes_evento"]).size().unstack(["uf", "grupo"], fill_value=0).to_string())

RS 2024, enchente — municípios só com os 5 códigos do enunciado: 56
RS 2024, enchente — municípios incluindo 13214 (Chuvas Intensas): 469 de 497

uf                   AM            RS     
grupo          enchente seca enchente seca
ano_mes_evento                            
2021-10               0    0        0    2
2021-11               0    0        0    2
2021-12               0    0        0   18
2022-01               1    0        2  268
2022-02               2    0        0   24
2022-03               9    0        2    5
2022-04              11    0        5    2
2022-05              31    0       12    0
2022-06               6    0        8    0
2022-07               0    0        5    0
2022-08               0    0        1    0
2022-09               1    1        0    0
2022-10               2   18        0    0
2022-11               1    3        1    3
2022-12               0    0        4   20
2023-01               0    0        2  190
2023-02               0    0        2

## 5. Salvar a tabela tratada e registrar a proveniência

In [7]:
SAIDA = TRATADOS / "atlas_eventos_am_rs_2022_2024"
tab.to_csv(SAIDA.with_suffix(".csv"), index=False)
tab.to_parquet(SAIDA.with_suffix(".parquet"), index=False)
print("Salvo:", SAIDA.with_suffix(".csv").relative_to(RAIZ), "e .parquet —", len(tab), "linhas")

registrar(
    "Atlas Digital de Desastres no Brasil (MIDR/Sedec, S2iD)",
    "https://atlasdigital.mdr.gov.br",
    "Download manual da planilha completa (Downloads) e leitura com pd.read_excel",
    {"arquivo": ARQ_ATLAS.name, "aba": ABA, "ufs": UFS, "anos": list(ANOS),
     "cobrade": sorted(COBRADE), "n_registros_tratados": len(tab)},
    len(atlas), ARQ_ATLAS,
)

Salvo: data/processed/atlas_eventos_am_rs_2022_2024.csv e .parquet — 2117 linhas
  ✓ proveniência: Atlas Digital de Desastres no Brasil (MIDR/Sedec, S2iD) (76190 linhas)


## 6. Limitações

* **O Atlas não tem data de fim.** Cada linha é o decreto/registro; não dá para saber quando a seca ou a enchente acabou. "Meses desde o início" é calculável, mas a duração do evento terá de ser uma regra assumida na EDA.
* **`Registro` ≠ `Reconhecido`.** Cerca de 13% das linhas filtradas são só `Registro`, sem reconhecimento federal. A EDA deve decidir se usa as duas ou só as reconhecidas.
* **Município sem linha ≠ município sem seca/enchente.** Só entra quem fez o registro no S2iD; é subnotificação possível, mais provável em municípios pequenos e pobres, justamente o foco da Big Idea.
* **Sobreposição de eventos.** O RS teve estiagem em 2022 e enchentes em 2023 e 2024; um município pode ter várias linhas no mesmo mês ou em tipos diferentes.
* Os valores de danos humanos vêm da aba `Atlas Valores Originais`. A aba `Valores Corrigidos` só corrige valores monetários e não é usada aqui.